# OceanSAR execution notebook
Alternative to command line execution

In [ ]:
#%load_ext autoreload
#%autoreload 2

In [ ]:
import sys
import os
import time
from drama.io import cfg as drcfg
#import stereoid.utils.config as st_config
from oceansar import ocs_io as osrio
from oceansar import utils
from oceansar.radarsim.sar_raw_nompi import sar_raw
from oceansar.radarsim.sar_processor import sar_focus
from oceansar.radarsim.ati_processor import ati_process
from oceansar.radarsim.insar_processor import insar_process
from oceansar.radarsim.L2_wavespectrum import l2_wavespectrum

# Configuration
Specifiy config file, overide some things in there which are pointless

In [ ]:
cfg_file = "/Users/paco/Documents/CODE/oceansar/par/sim_example_rosel.cfg"
cfg = drcfg.ConfigFile(cfg_file)
do_raw = cfg.sim.raw_run
raw_file = os.path.join(cfg.sim.path, cfg.sim.raw_file)
ocean_file = os.path.join(cfg.sim.path, cfg.sim.ocean_file)
errors_file = os.path.join(cfg.sim.path, cfg.sim.errors_file)
do_focus = cfg.sim.proc_run
slc_file = os.path.join(cfg.sim.path, cfg.sim.proc_file)
if cfg.sim.insar_run or cfg.sim.ati_run:
    do_insar = True
    insar_file = os.path.join(cfg.sim.path, cfg.sim.insar_file)
if  cfg.sim.ati_run:
    do_ati = True   
    ati_file = os.path.join(cfg.sim.path, cfg.sim.ati_file)
do_spectra = cfg.sim.L2_wavespectrum_run   
xspectra_file = os.path.join(cfg.sim.path, cfg.sim.xspectra_file)


# Run oceansar
...

In [ ]:
# Execute components of oceansar
# Create output directory if it doesnt exist already
os.makedirs(cfg.sim.path, exist_ok=True)
if do_raw:
    print('Launching SAR RAW Generator...')
    sar_raw(cfg.cfg_file_name, raw_file, ocean_file,
            cfg.sim.ocean_reuse,
            errors_file, cfg.sim.errors_reuse, plot_save=True)
    
if do_focus:
    print('Launching SAR Focusing Processor...')
    sar_focus(cfg.cfg_file_name, raw_file, slc_file)

if do_insar:
    print('Launching InSAR Processor...')
    print('Launching InSAR L1b Processor...')
    insar_process(cfg.cfg_file_name,
                  slc_file, ocean_file, insar_file)
    
if do_ati:
    print('Launching ATI Processor...')
    ati_process(cfg.cfg_file_name, insar_file, ocean_file, ati_file)    

if do_spectra:
    print('Launching L2 Wave Spectrum Processor...')
    l2_wavespectrum(cfg.cfg_file_name, slc_file, ocean_file, xspectra_file)


In [ ]:
%load_ext line_profiler
def profile_sar_raw():
    return sar_raw(
        cfg.cfg_file_name,
        raw_file,
        ocean_file,
        cfg.sim.ocean_reuse,
        errors_file,
        cfg.sim.errors_reuse,
        plot_save=True,
    )
%lprun -f sar_raw profile_sar_raw()

In [ ]:
# GeoHistory range-history diagnostic
import inspect
import numpy as np
import matplotlib.pyplot as plt
from scipy import constants as const
from drama.geo.geo_history import GeoHistory
from drama.performance.sar.sar_performance_common import calc_analysis_time
from oceansar.utils import geometry as geosar

inc = np.deg2rad(cfg.sar.inc_angle)
alt = cfg.orbit.Horb
v_ground = (geosar.orbit_to_vel(alt, ground=True)
            if cfg.sar.v_ground == 'auto' else cfg.sar.v_ground)
sr0 = geosar.inc_to_sr(inc, alt)
gr0 = geosar.inc_to_gr(inc, alt)
la0 = np.asarray(geosar.gr_to_geo(np.array([gr0]), alt)[2]).item()
t_span = (1.5 * sr0 * (const.c / cfg.sar.f0) / cfg.sar.ant_L_tx
          + cfg.ocean.Ly) / v_ground
t_analysis = t_span + calc_analysis_time(
    alt, inc, cfg.sar.f0, cfg.sar.prf, n_amb=1)

gh_test = GeoHistory(
    cfg, latitude=10,
    inc_range=np.degrees([inc, inc + np.deg2rad(30)]) + [-3, 3],
    inc_swth=np.degrees(inc) + np.array([-1, 1]),
    n_la_pts=800, t_analysis=t_analysis, aei=None)

t_required = 0.5 * (t_span + cfg.ocean.Ly / v_ground)
t = np.linspace(-t_required, t_required, 1001)
sr_gh = gh_test.sr_spl(la0, t).ravel()
sr_gh -= gh_test.sr_spl(la0, 0).item()
sr_quadratic = (v_ground * t) ** 2 / (2 * sr0)

print('GeoHistory loaded from:', inspect.getfile(GeoHistory))
print(f'Required time range: [{t.min():.3f}, {t.max():.3f}] s')
print(f'Spline time range:   [{gh_test.t.min():.3f}, {gh_test.t.max():.3f}] s')
print('Outside spline domain:', t.min() < gh_test.t.min() or t.max() > gh_test.t.max())
print(f'Max |GeoHistory - quadratic|: {np.max(np.abs(sr_gh - sr_quadratic)):.3f} m')

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(t, sr_gh, label='GeoHistory')
ax[0].plot(t, sr_quadratic, '--', label='quadratic approximation')
ax[0].set(xlabel='Azimuth time [s]', ylabel='Range offset [m]', title='Range history')
ax[0].legend()
ax[0].grid(True)
ax[1].plot(t, sr_gh - sr_quadratic)
ax[1].set(xlabel='Azimuth time [s]', ylabel='Difference [m]', title='GeoHistory minus approximation')
ax[1].grid(True)
plt.tight_layout()


In [ ]:
la0